### Create batch of prompts

#### Version 1 -> tasks (13) x topics (49) = 637 questions

In [17]:
import json
from tqdm import tqdm
from multiturn_modul import MultiturnStyle, get_multiturn_style
import pandas as pd
import random

In [ ]:
# Create batch of prompts

# Specify which model should be used to answer prompts
gpt_model = "gpt-4o"

# Path for input data and to store prompts
path_medical_ai_tasks = "../resources/medical_ai_tasks.json"
path_medical_topics = "../resources/medical_topics.json"
path_moove_examples = "../resources/generated_doctors_questions.jsonl"

output_path = "../results/batched_prompts_test.jsonl"

# Load data
with open(path_medical_ai_tasks, "r") as f:
    medical_ai_tasks = json.load(f)
with open(path_medical_topics, "r") as f:
    medical_topics = json.load(f)
with open(path_moove_examples, "r") as f:
    moove_examples = [json.loads(line) for line in f]
# Define number of few-shoot examples to be added
num_examples = 2

# Helper function to get uniformly at random num_entries moove examples
def get_random_entries(moove_examples, num_examples=2):
    """
    Returns a specified number of random entries from moove_examples without replacement.

    param moove_examples: List of JSON objects loaded from a .jsonl file.
    param num_entries: Number of random entries to return.
    return: List of randomly selected entries.
    """
    if len(moove_examples) < num_examples:
        raise ValueError(f"Not enough entries in moove_examples to select {num_examples} unique entries.")
    
    return random.sample(moove_examples, num_examples)



# prompt and context
content = "You are an assistant responsible for creating prompts that healthcare workers would ask a medical AI chatbot."
def get_prompt(task, description, additional_instruction, topic, num_examples, example_1, example_2):
    prompt = f'''Generate a prompt that a physician might ask an AI chatbot when tasked with "{task}" in the context of the medical topic "{topic}".
{task} is described as: {description}
To create a realistic prompt, follow these additional instructions: {additional_instruction}
Only include the generated prompt, adding extra details only if explicitly instructed. Focus solely on generating a realistic prompt a physician might ask a medical AI chatbot.
Below there are {num_examples} examples of real prompts that physicians have previously asked to the medical AI chatbot:
{example_1}
{example_2}
The examples provided are likely not directly related to "{task}" in the context of "{topic}", but they are representative of the format and style physicians use. The prompt you generate should align with the format and style of the {num_examples} examples provided above.'''
    return prompt


print("Creating a batch of prompts")
print(f"that can be processed by {gpt_model} in batch mode")

with open(output_path, 'w') as file:
    count = 0
    for ai_task in tqdm(medical_ai_tasks):
        task = ai_task["task"]
        description = ai_task["description"]
        max_token = ai_task["max_token"]
        additional_instruction = ai_task["additional_instruction"]
        for med_topic in medical_topics:
            topic = med_topic["topic"]
            content = content
            example_1, example_2 = get_random_entries(moove_examples, num_examples)
            prompt = get_prompt(task, description, additional_instruction, topic, num_examples, example_1["question"], example_2["question"])
            line = {
                "custom_id": str(count) + "-" + task + "-" + topic,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": gpt_model,
                    "messages": [
                        {"role": "system", "content": content},
                        {"role": "user", "content": f"{prompt}"}
                    ],
                    "max_tokens": max_token
                }
            }
            file.write(json.dumps(line) + '\n')
            count += 1

print(f"batch of {count} prompts saved to {output_path}")
print(f"See below an example prompt that will be processed by {gpt_model}:")
print("*******************************************************************")
print(prompt)
print("*******************************************************************")

Creating a batch of prompts
that can be processed by gpt-4o in batch mode


100%|██████████| 13/13 [00:00<00:00, 307.87it/s]

batch of 637 prompts saved to ../results/batched_prompts_test.jsonl
See below an example prompt that will be processed by gpt-4o:
*******************************************************************
Generate a prompt that a physician might ask an AI chatbot when tasked with "Health Policy Guidance" in the context of the medical topic "Preventive Medicine".
Health Policy Guidance is described as: Assisting with compliance to regulations and policies in healthcare.
To create a realistic prompt, follow these additional instructions: Provide a brief example of a healthcare regulation scenario, focusing on policy compliance.
Only include the generated prompt, adding extra details only if explicitly instructed. Focus solely on generating a realistic prompt a physician might ask a medical AI chatbot.
Below there are 2 examples of real prompts that physicians have previously asked to the medical AI chatbot:
As a primary care physician in a metropolitan area, I have a 45-year-old patient with a 